In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime

# ══════════════════════════════════════════════════════════════════════════════
#  FIDELITY-TRADABLE ASSET UNIVERSE
#  Covers every major asset class available on Fidelity:
#    US Equities  — 11 GICS sectors
#    ETFs         — Broad market, sector, international, fixed income,
#                   REIT, dividend, factor
#    Metals       — Precious-metal ETFs, miners, base/industrial metals
#    Currencies   — Currency ETFs (Fidelity does not offer spot forex)
#    Commodities  — Energy, agriculture, diversified
#    Crypto       — Fidelity-listed spot BTC/ETH ETFs (approved 2024)
# ══════════════════════════════════════════════════════════════════════════════

# ── US Equities: 11 GICS Sectors (~30 stocks each) ───────────────────────────
EQUITIES = {
    "Communication Services": [
        "GOOG","META","DIS","NFLX","T","VZ","CMCSA","TMUS","CHTR","EA",
        "TTWO","LYV","WBD","ROKU","SNAP","PINS","SPOT","OMC","NWSA","FOXA",
        "SIRI","NXST","BIDU","NTES","SE","BILI","ZM","VOD","AMX","LUMN",
    ],
    "Consumer Discretionary": [
        "AMZN","TSLA","HD","MCD","NKE","LOW","SBUX","TJX","BKNG","CMG",
        "GM","F","ROST","MAR","ORLY","YUM","AZO","DHI","LEN","RCL",
        "HLT","DG","KMX","BBY","ETSY","HAS","MGM","DRI","ULTA","POOL",
    ],
    "Consumer Staples": [
        "PG","KO","PEP","WMT","PM","MO","COST","MDLZ","CL","GIS",
        "KHC","KMB","STZ","ADM","MNST","EL","TAP","CPB","CAG","HSY",
        "TSN","MKC","CHD","KDP","CLX","HRL","SJM","KR","SYY","BG",
    ],
    "Energy": [
        "XOM","CVX","COP","EOG","SLB","PSX","VLO","MPC","OXY","HES",
        "DVN","FANG","HAL","BKR","MRO","APA","CTRA","RRC","AR","EQT",
        "KMI","WMB","TRGP","OKE","LNG","PBR","SU","OVV","PAA","NOG",
    ],
    "Financials": [
        "JPM","BAC","WFC","C","GS","MS","BLK","SCHW","AXP","USB",
        "PNC","TFC","COF","BK","STT","CB","AIG","MET","PRU","ALL",
        "CME","ICE","SPGI","MCO","NDAQ","V","MA","PYPL","AON","SYF",
    ],
    "Health Care": [
        "JNJ","UNH","LLY","PFE","ABBV","MRK","TMO","ABT","DHR","BMY",
        "AMGN","GILD","CVS","ISRG","SYK","ZTS","REGN","VRTX","BSX","BDX",
        "EW","IQV","MCK","CI","HCA","IDXX","RMD","HOLX","LH","DGX",
    ],
    "Industrials": [
        "HON","UNP","UPS","RTX","CAT","BA","DE","GE","MMM","LMT",
        "FDX","NOC","GD","EMR","ETN","CSX","WM","CMI","WAB","LHX",
        "GWW","PH","ROP","TT","DAL","AAL","UAL","FAST","CTAS","URI",
    ],
    "Information Technology": [
        "AAPL","MSFT","NVDA","AVGO","ORCL","CSCO","ACN","ADBE","TXN","IBM",
        "CRM","AMD","INTC","QCOM","AMAT","INTU","ADI","MU","LRCX","NOW",
        "CDNS","SNPS","KLAC","NXPI","PANW","FTNT","WDAY","HPQ","STX","APH",
    ],
    "Materials": [
        "LIN","SHW","APD","ECL","NEM","FCX","DD","VMC","MLM","LYB",
        "PPG","DOW","NUE","IFF","ALB","MOS","CF","FMC","AVY","CLF",
        "RS","STLD","IP","PKG","OC","OLN","RPM","EMN","AA","SQM",
    ],
    "Real Estate": [
        "PLD","AMT","EQIX","SPG","PSA","CCI","WELL","O","DLR","EXR",
        "AVB","EQR","ESS","VTR","CUBE","MAA","INVH","UDR","SUI","CPT",
        "KIM","FRT","IRM","HST","NNN","STAG","BXP","REXR","ADC","ARE",
    ],
    "Utilities": [
        "NEE","SO","DUK","D","AEP","EXC","SRE","XEL","PEG","ED",
        "WEC","ES","EIX","DTE","PPL","FE","AEE","CMS","ATO","NI",
        "CNP","EVRG","LNT","PNW","AES","NRG","VST","AWK","HE","UGI",
    ],
}

# ── ETFs ──────────────────────────────────────────────────────────────────────
ETFS = {
    "ETF - Broad Market": [
        "SPY","VOO","IVV","QQQ","QQQM","IWM","DIA","VTI","ITOT",
        "MDY","IJH","IJR","VXF",
    ],
    "ETF - Sector": [
        "XLC","XLY","XLP","XLE","XLF","XLV","XLI","XLK","XLB","XLRE","XLU",
    ],
    "ETF - International": [
        "EFA","EEM","VEA","VWO","IEFA","EWJ","EWZ","FXI","EWG","EWU",
        "EWC","INDA","VGK","EWY","EWA","MCHI","ACWI","ACWX",
    ],
    "ETF - Fixed Income": [
        "TLT","IEF","SHY","AGG","BND","LQD","HYG","JNK","TIP","SCHP",
        "VCIT","VCSH","MUB","EMB","BNDX","GOVT","BSV","BIV","BLV",
    ],
    "ETF - REIT": [
        "VNQ","IYR","SCHH","REM","MORT",
    ],
    "ETF - Dividend & Factor": [
        "VIG","VYM","NOBL","HDV","DGRO","DVY",
        "IWF","IWD","MTUM","QUAL","VLUE","USMV",
    ],
}

# ── Metals ────────────────────────────────────────────────────────────────────
METALS = {
    "Metal - Precious": [
        "GLD","IAU","SLV","PPLT","PALL","BAR","SIVR",
    ],
    "Metal - Miners": [
        "GDX","GDXJ","SIL","SILJ","NEM","GOLD","AEM","WPM","FNV","RGLD",
    ],
    "Metal - Base & Industrial": [
        "COPX","CPER","PICK","REMX","LIT","BATT",
    ],
}

# ── Currencies ────────────────────────────────────────────────────────────────
# Fidelity does not offer spot forex — currency exposure via ETFs
CURRENCIES = {
    "Currency ETFs": [
        "UUP","UDN","FXE","FXY","FXB","FXC","FXA","FXF","CEW","CYB",
    ],
}

# ── Commodities ───────────────────────────────────────────────────────────────
COMMODITIES = {
    "Commodity - Energy":      ["USO","BNO","UNG","BOIL"],
    "Commodity - Agriculture": ["DBA","CORN","WEAT","SOYB","CANE","JO"],
    "Commodity - Diversified": ["DBC","PDBC","GSG","COMT"],
}

# ── Crypto ETFs (Fidelity spot ETFs, approved 2024) ───────────────────────────
CRYPTO = {
    "Crypto ETFs": ["FBTC","IBIT","FETH","ETHA","BTCO","BITB"],
}

# ── Merge all asset classes ───────────────────────────────────────────────────
ASSET_CLASSES = {}
ASSET_CLASSES.update(EQUITIES)
ASSET_CLASSES.update(ETFS)
ASSET_CLASSES.update(METALS)
ASSET_CLASSES.update(CURRENCIES)
ASSET_CLASSES.update(COMMODITIES)
ASSET_CLASSES.update(CRYPTO)

# Deduplicate while preserving first-seen assignment
seen = set()
ALL_TICKERS = []
ASSET_MAP   = {}
for cls, tickers in ASSET_CLASSES.items():
    for t in tickers:
        if t not in seen:
            ALL_TICKERS.append(t)
            seen.add(t)
        ASSET_MAP[t] = cls

start_date = "2021-01-01"
end_date   = datetime.now().strftime("%Y-%m-%d")

print(f"Total unique tickers : {len(ALL_TICKERS)}")
print(f"Asset classes        : {len(ASSET_CLASSES)}")
for cls, tickers in ASSET_CLASSES.items():
    print(f"  {cls:<35} {len(tickers):>3} tickers")


In [4]:
import pandas as pd
import yfinance as yf
import time

# 1. Load the original file that you have on your computer
print("Loading nasdaq-listed.csv...")
df = pd.read_csv('nasdaq-listed.csv').dropna()

# 2. Filter out non-stock assets and categorize into 11 sectors
print("Filtering non-stock assets and assigning sectors...")

exclude_keywords = [
    r'ETF', r'Warrant', r'Right', r'Unit', r'Preferred', 
    r'Depositary', r'Depository', r'ADR', r'ADS', r'Note', r'Bond', 
    r'Bull \dX', r'Bear \dX', r'Daily ETF', r'Index', r'Trust', r'Debt', 
    r'Receipt', r'Fund', r'Portfolio', r'TEST STOCK', r'Test Stock'
]
mask = df['Security Name'].str.contains('|'.join(exclude_keywords), case=False, na=False)
stocks_only = df[~mask].copy()

# Sector mapping based on company names
sector_mapping = {
    'Health Care': r'Bio|Pharma|Therap|Medical|Health|Life Science|Clinic|Oncology|Genetic|Diagnostic|Medicine|Neuro|Immuno|Vaccine|Care|Geno|Surge|Cardio|Dental',
    'Utilities': r'Utility|Utilities|Water|Electric|Light|Power',
    'Materials': r'Gold|Silver|Mining|Material|Steel|Copper|Chemical|Lithium|Metal|Mineral|Plastic|Packaging|Paper',
    'Real Estate': r'Real Estate|REIT|Property|Properties|Home|Realty|Estate|Housing|Apartment',
    'Energy': r'Energy|Oil|Gas|Petroleum|Resource|Solar|Fuel|Midstream|Drilling|Pipeline|Clean|Renew|Wind|Carbon',
    'Information Technology': r'Tech|Software|Cyber|Network|Computing|Semiconductor|Micro|AI|Data|System|Cloud|Digital|Info|IT|Solution|Opto|Device|Instrument|Electronic|Science|Innovat|Intelligen',
    'Communication Services': r'Media|Communication|Telecom|Wireless|Broadcast|Interactive|Cable|Internet|Publishing|Satellite|Radio|Television|Social',
    'Industrials': r'Airline|Freight|Logistic|Manufacturing|Aerospace|Defense|Transport|Machinery|Aviation|Industrial|Marine|Shipping|Equipment|Building|Construct|Aero|Vehicle|Engine',
    'Consumer Staples': r'Food|Beverage|Farm|Grocery|Nutrition|Agri|Brew|Supermarket|Drink|Meat',
    'Consumer Discretionary': r'Auto|Entertain|Leisure|Retail|Apparel|Brand|Hospitality|Hotel|Resort|Motor|Restaurant|Education|Consumer|Gaming|Sport|Tour|Fashion|Beauty',
    'Financials': r'Bank|Bancorp|Financial|Trust|Capital|Holding|Insurance|Equity|Fund|Credit|Mortgage|Acquisition|Invest|Partner|Wealth|Bancshares|L.P.|Corp|Group|Inc|Company|Ltd'
}

stocks_only['Sector'] = 'Financials' 
for sector_name, pattern in sector_mapping.items():
    s_mask = stocks_only['Security Name'].str.contains(pattern, case=False, na=False)
    stocks_only.loc[s_mask, 'Sector'] = sector_name

explicit_map = {
    'AAPL': 'Information Technology', 'MSFT': 'Information Technology', 'GOOG': 'Communication Services',
    'GOOGL': 'Communication Services', 'AMZN': 'Consumer Discretionary', 'TSLA': 'Consumer Discretionary',
    'META': 'Communication Services', 'NVDA': 'Information Technology', 'NFLX': 'Communication Services',
    'PEP': 'Consumer Staples', 'COST': 'Consumer Staples', 'CSCO': 'Information Technology'
}
for sym, sec in explicit_map.items():
    if sym in stocks_only['Symbol'].values:
        stocks_only.loc[stocks_only['Symbol'] == sym, 'Sector'] = sec

tickers = stocks_only['Symbol'].tolist()

# 3. Download data in batches to avoid rate limits
print(f"Downloading historical data from 2022-01-01 for {len(tickers)} stocks in batches...")

batch_size = 100
all_data = []

for i in range(0, len(tickers), batch_size):
    batch_tickers = tickers[i:i + batch_size]
    print(f"Downloading batch {i // batch_size + 1} of {(len(tickers) // batch_size) + 1}...")
    
    # Download the batch
    batch_data = yf.download(batch_tickers, start='2022-01-01', threads=True)['Close']
    all_data.append(batch_data)
    
    # Pause for 2 seconds to keep Yahoo happy
    time.sleep(2)

# Combine all batches into one dataframe
data = pd.concat(all_data, axis=1)

max_trading_days = len(data)
print(f"\nTotal expected trading days: {max_trading_days}")

# 4. Filter for stocks that have complete data
valid_tickers = []
for ticker in data.columns:
    # Ensure it's a pandas Series (sometimes single-ticker batches return differently)
    if isinstance(data[ticker], pd.Series):
        series = data[ticker].dropna()
        if len(series) > 0:
            first_date = series.index[0]
            if first_date <= pd.Timestamp('2022-01-10') and len(series) >= (max_trading_days * 0.95):
                valid_tickers.append(ticker)

print(f"Found {len(valid_tickers)} stocks with complete, active trading data since 2022.")

# 5. Save the final list to a new CSV
final_df = stocks_only[stocks_only['Symbol'].isin(valid_tickers)]
output_filename = 'nasdaq_complete_data_2022_to_present_FULL.csv'
final_df.to_csv(output_filename, index=False)

print(f"Success! Saved the final list to {output_filename}")

Loading nasdaq-listed.csv...
Filtering non-stock assets and assigning sectors...


C:\Users\ibrah\AppData\Local\Temp\ipykernel_13784\238246157.py:64: FutureWarning: YF.download() has changed argument auto_adjust default to True
  batch_data = yf.download(batch_tickers, start='2022-01-01', threads=True)['Close']
[*********************100%***********************]  100 of 100 completed


C:\Users\ibrah\AppData\Local\Temp\ipykernel_13784\238246157.py:64: FutureWarning: YF.download() has changed argument auto_adjust default to True
  batch_data = yf.download(batch_tickers, start='2022-01-01', threads=True)['Close']
[*********************100%***********************]  100 of 100 completed


C:\Users\ibrah\AppData\Local\Temp\ipykernel_13784\238246157.py:64: FutureWarning: YF.download() has changed argument auto_adjust default to True
  batch_data = yf.download(batch_tickers, start='2022-01-01', threads=True)['Close']
[*********************100%***********************]  100 of 100 completed

1 Failed download:
['ATMC']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


C:\Users\ibrah\AppData\Local\Temp\ipykernel_13784\238246157.py:64: FutureWarning: YF.download() has changed argument auto_adjust default to True
  batch_data = yf.download(batch_tickers, start='2022-01-01', threads=True)['Close']
[*********************100%***********************]  100 of 100 completed


...

C:\Users\ibrah\AppData\Local\Temp\ipykernel_13784\238246157.py:64: FutureWarning: YF.download() has changed argument auto_adjust default to True
  batch_data = yf.download(batch_tickers, start='2022-01-01', threads=True)['Close']
[*********************100%***********************]  100 of 100 completed


C:\Users\ibrah\AppData\Local\Temp\ipykernel_13784\238246157.py:64: FutureWarning: YF.download() has changed argument auto_adjust default to True
  batch_data = yf.download(batch_tickers, start='2022-01-01', threads=True)['Close']
[*********************100%***********************]  100 of 100 completed


C:\Users\ibrah\AppData\Local\Temp\ipykernel_13784\238246157.py:64: FutureWarning: YF.download() has changed argument auto_adjust default to True
  batch_data = yf.download(batch_tickers, start='2022-01-01', threads=True)['Close']
[*********************100%***********************]  100 of 100 completed


C:\Users\ibrah\AppData\Local\Temp\ipykernel_13784\238246157.py:64: FutureWarning: YF.download() has changed argument auto_adjust default to True
  batch_data = yf.download(batch_tickers, start='2022-01-01', threads=True)['Close']
HTTP Error 404: *******86%****************       ]  79 of 92 completed
[*********************100%***********************]  92 of 92 completed

1 Failed download:
['ZXYZ.A']: YFTzMissingError('possibly delisted; no timezone found')



Total expected trading days: 1037
Found 2360 stocks with complete, active trading data since 2022.
Success! Saved the final list to nasdaq_complete_data_2022_to_present_FULL.csv


In [ ]:
import time

# ── Download all asset classes and save one CSV per class ─────────────────────
# Requires ASSET_CLASSES, ALL_TICKERS, ASSET_MAP, start_date, end_date
# defined in the cell above.

for asset_class, tickers in ASSET_CLASSES.items():
    class_data = pd.DataFrame()
    safe_name  = asset_class.replace(" ", "_").replace("-", "").replace("&", "and")

    for ticker in tickers:
        try:
            df = yf.Ticker(ticker).history(start=start_date, end=end_date, interval="1d")
            if not df.empty:
                df["Ticker"]      = ticker
                df["AssetClass"]  = asset_class
                class_data = pd.concat([class_data, df])
            else:
                print(f"  [no data] {ticker}")
        except Exception as e:
            print(f"  [error]   {ticker}: {e}")

        time.sleep(0.05)   # gentle rate-limit

    if not class_data.empty:
        fname = f"{safe_name}.csv"
        class_data.to_csv(fname, index=True)
        print(f"✅  {asset_class:<35} → {fname}  ({len(class_data):,} rows)")
    else:
        print(f"⚠️  {asset_class} — no data collected, CSV skipped")

print("\n🚀  Download complete.")
print(f"    {len(ASSET_CLASSES)} asset-class CSV files written.")
print(f"    Total unique tickers attempted: {len(ALL_TICKERS)}")
